# Data Preprocessing

## About this dataset

This dataset contains 7,668 movies with 15 features capturing a mix of content, performance, and production-related attributes. It provides a structured view of films across multiple decades, making it well-suited for both exploratory analysis and machine learning applications.

Each record includes key information such as:

Content attributes: genre, director, writer, and lead actor
Performance metrics: user rating (score) and number of votes
Financial data: production budget and box office gross
Contextual details: release date, country, runtime, and production company

The dataset combines categorical, numerical, and text-based fields, enabling a wide range of use cases including:

content-based recommendation systems
predictive modeling and ranking
feature engineering and data preprocessing workflows

While the dataset does not include user interaction data (e.g., clicks or watch history), it provides rich item-level features that are ideal for building content-driven recommendation systems and machine learning ranking models.

## Data loading + exploring

In [2]:
import pandas as pd
import numpy as np
from numpy.linalg import norm
import os

data = pd.read_csv("../data/raw/movies.csv")
data

,name,rating,genre,year,released,score,votes,director,writer,star,country,budget,gross,company,runtime
0,The Shining,R,Drama,1980,"June 13, 1980 (United States)",8.4,927000.0,Stanley Kubrick,Stephen King,Jack Nicholson,United Kingdom,19000000.0,46998772.0,Warner Bros.,146.0
1,The Blue Lagoon,R,Adventure,1980,"July 2, 1980 (United States)",5.8,65000.0,Randal Kleiser,Henry De Vere Stacpoole,Brooke Shields,United States,4500000.0,58853106.0,Columbia Pictures,104.0
2,Star Wars: Episode V - The Empire Strikes Back,PG,Action,1980,"June 20, 1980 (United States)",8.7,1200000.0,Irvin Kershner,Leigh Brackett,Mark Hamill,United States,18000000.0,538375067.0,Lucasfilm,124.0
3,Airplane!,PG,Comedy,1980,"July 2, 1980 (United States)",7.7,221000.0,Jim Abrahams,Jim Abrahams,Robert Hays,United States,3500000.0,83453539.0,Paramount Pictures,88.0
4,Caddyshack,R,Comedy,1980,"July 25, 1980 (United States)",7.3,108000.0,Harold Ramis,Brian Doyle-Murray,Chevy Chase,United States,6000000.0,39846344.0,Orion Pictures,98.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7663,More to Life,NaN,Drama,2020,"October 23, 2020 (United States)",3.1,18.0,Joseph Ebanks,Joseph Ebanks,Shannon Bond,United States,7000.0,NaN,NaN,90.0
7664,Dream Round,NaN,Comedy,2020,"February 7, 2020 (United States)",4.7,36.0,Dusty Dukatz,Lisa Huston,Michael Saquella,United States,NaN,NaN,Cactus Blue Entertainment,90.0
7665,Saving Mbango,NaN,Drama,2020,"April 27, 2020 (Cameroon)",5.7,29.0,Nkanya Nkwai,Lynno Lovert,Onyama Laura,United States,58750.0,NaN,Embi Productions,NaN
7666,It's Just Us,NaN,Drama,2020,"October 1, 2020 (United States)",NaN,NaN,James Randall,James Randall,Christina Roz,United States,15000.0,NaN,NaN,120.0


In [3]:
df = data.copy()

### Understanding columns

In [4]:
df.groupby("country").size().sort_values(ascending=False)

country
United States                     5475
United Kingdom                     816
France                             279
Canada                             190
Germany                            117
Australia                           92
Japan                               81
India                               62
Italy                               61
Spain                               47
Hong Kong                           45
Ireland                             43
China                               40
South Korea                         35
Denmark                             32
New Zealand                         25
Sweden                              25
Mexico                              22
Norway                              12
West Germany                        12
Netherlands                         12
Iran                                10
Switzerland                         10
South Africa                         8
Czech Republic                       8
Russia           

In [5]:
df.groupby("year").size()

year
1980     92
1981    113
1982    126
1983    144
1984    168
1985    200
1986    200
1987    200
1988    200
1989    200
1990    200
1991    200
1992    200
1993    200
1994    200
1995    200
1996    200
1997    200
1998    200
1999    200
2000    200
2001    200
2002    200
2003    200
2004    200
2005    200
2006    200
2007    200
2008    200
2009    200
2010    200
2011    200
2012    200
2013    200
2014    200
2015    200
2016    200
2017    200
2018    200
2019    200
2020     25
dtype: int64

In [6]:
df.groupby("director").size().sort_values(ascending=False)

director
Woody Allen             38
Clint Eastwood          31
Directors               28
Steven Spielberg        27
Ron Howard              24
                        ..
Jeff Baena               1
Jefery Levy              1
Jeb Stuart               1
Jean-François Richet     1
Éva Gárdos               1
Length: 2949, dtype: int64

In [7]:
df.groupby("genre").size().sort_values(ascending=False)

genre
Comedy       2245
Action       1705
Drama        1518
Crime         551
Biography     443
Adventure     427
Animation     338
Horror        322
Fantasy        44
Mystery        20
Thriller       16
Family         11
Romance        10
Sci-Fi         10
Western         3
Musical         2
Sport           1
Music           1
History         1
dtype: int64

## Data Cleaning

In [8]:
df = df.dropna(axis=0)

In [9]:
# gets rid of all rows with votes kess than 5000
df = df.loc[df["votes"] > 5000, ["name", "genre", "year", "score", "votes", "director",	"writer", "star", "country",	"gross",	"runtime"]]

In [10]:
# creates new column which combines score and votes

df["popularityScore"] = df["score"] * df["votes"]

In [11]:
# deletes unwanted columns from dataset

del df["score"]
del df["votes"]

In [12]:
# buckets
# makes director, write and star have a value from 0-5 on how popular they are grouped with their name

df["director"] = df["director"].apply(lambda x: (x, (df[df["director"] == x]["director"].count()) // 10))
df["writer"] = df["writer"].apply(lambda x: (x, (df[df["writer"] == x]["writer"].count()) // 10))
df["star"] = df["star"].apply(lambda x: (x, (df[df["star"] == x]["star"].count()) // 10))

In [13]:
# removes all countrys with less than 8 occurences

df = df[df.groupby('country')['country'].transform('count') > 8]

In [14]:
# create buckets for countries around the world

def map_country(country):
    if country in ["United States", "Canada"]:
        return "USCan"
    elif country in ["United Kingdom", "Ireland"]:
        return "GB"
    elif country in ["Australia", "New Zealand"]:
        return "Oceania"
    elif country in ["France", "Germany", "Spain", "Italy", "Denmark", "Sweden", "Norway", "Netherlands"]:
        return "EU"
    elif country in ["Japan", "Hong Kong", "China", "South Korea"]:
        return "East Asia"
    else:
        return "Other"

df["country"] = df["country"].apply(map_country)

In [15]:
df.groupby("country").size()

country
EU            224
East Asia      85
GB            467
Oceania        59
Other          10
USCan        4092
dtype: int64

In [16]:
df

,name,genre,year,director,writer,star,country,gross,runtime,popularityScore
0,The Shining,Drama,1980,"(Stanley Kubrick, 0)","(Stephen King, 2)","(Jack Nicholson, 1)",GB,46998772.0,146.0,7786800.0
1,The Blue Lagoon,Adventure,1980,"(Randal Kleiser, 0)","(Henry De Vere Stacpoole, 0)","(Brooke Shields, 0)",USCan,58853106.0,104.0,377000.0
2,Star Wars: Episode V - The Empire Strikes Back,Action,1980,"(Irvin Kershner, 0)","(Leigh Brackett, 0)","(Mark Hamill, 0)",USCan,538375067.0,124.0,10440000.0
3,Airplane!,Comedy,1980,"(Jim Abrahams, 0)","(Jim Abrahams, 0)","(Robert Hays, 0)",USCan,83453539.0,88.0,1701700.0
4,Caddyshack,Comedy,1980,"(Harold Ramis, 0)","(Brian Doyle-Murray, 0)","(Chevy Chase, 1)",USCan,39846344.0,98.0,788400.0
...,...,...,...,...,...,...,...,...,...,...
7646,The Invisible Man,Drama,2020,"(Leigh Whannell, 0)","(Leigh Whannell, 1)","(Elisabeth Moss, 0)",USCan,143151000.0,124.0,1320600.0
7648,Bad Boys for Life,Action,2020,"(Adil El Arbi, 0)","(Peter Craig, 0)","(Will Smith, 2)",USCan,426505244.0,124.0,924000.0
7649,Sonic the Hedgehog,Action,2020,"(Jeff Fowler, 0)","(Pat Casey, 0)","(Ben Schwartz, 0)",USCan,319715683.0,99.0,663000.0
7650,Dolittle,Adventure,2020,"(Stephen Gaghan, 0)","(Stephen Gaghan, 0)","(Robert Downey Jr., 1)",USCan,245487753.0,101.0,296800.0


### Save names for later

In [17]:
movie_names = []
for mov, year in zip(list(df["name"]), list(df["year"])):
    movie_names.append(f"{mov}({year})")

movie_names

['The Shining(1980)',
 'The Blue Lagoon(1980)',
 'Star Wars: Episode V - The Empire Strikes Back(1980)',
 'Airplane!(1980)',
 'Caddyshack(1980)',
 'Friday the 13th(1980)',
 'The Blues Brothers(1980)',
 'Raging Bull(1980)',
 'Superman II(1980)',
 'The Long Riders(1980)',
 'Any Which Way You Can(1980)',
 'Popeye(1980)',
 'Ordinary People(1980)',
 'Dressed to Kill(1980)',
 'Somewhere in Time(1980)',
 '9 to 5(1980)',
 'The Fog(1980)',
 "Heaven's Gate(1980)",
 'The Final Countdown(1980)',
 'Xanadu(1980)',
 'Brubaker(1980)',
 'American Gigolo(1980)',
 'Private Benjamin(1980)',
 'Motel Hell(1980)',
 'The Stunt Man(1980)',
 'Stardust Memories(1980)',
 'Bronco Billy(1980)',
 'The Octagon(1980)',
 'Indiana Jones and the Raiders of the Lost Ark(1981)',
 'An American Werewolf in London(1981)',
 'Escape from New York(1981)',
 'The Evil Dead(1981)',
 "Porky's(1981)",
 'Blow Out(1981)',
 'Clash of the Titans(1981)',
 'Excalibur(1981)',
 'Mad Max 2(1981)',
 'Stripes(1981)',
 'The Cannonball Run(1981)'

In [18]:
del df["name"]
df

,genre,year,director,writer,star,country,gross,runtime,popularityScore
0,Drama,1980,"(Stanley Kubrick, 0)","(Stephen King, 2)","(Jack Nicholson, 1)",GB,46998772.0,146.0,7786800.0
1,Adventure,1980,"(Randal Kleiser, 0)","(Henry De Vere Stacpoole, 0)","(Brooke Shields, 0)",USCan,58853106.0,104.0,377000.0
2,Action,1980,"(Irvin Kershner, 0)","(Leigh Brackett, 0)","(Mark Hamill, 0)",USCan,538375067.0,124.0,10440000.0
3,Comedy,1980,"(Jim Abrahams, 0)","(Jim Abrahams, 0)","(Robert Hays, 0)",USCan,83453539.0,88.0,1701700.0
4,Comedy,1980,"(Harold Ramis, 0)","(Brian Doyle-Murray, 0)","(Chevy Chase, 1)",USCan,39846344.0,98.0,788400.0
...,...,...,...,...,...,...,...,...,...
7646,Drama,2020,"(Leigh Whannell, 0)","(Leigh Whannell, 1)","(Elisabeth Moss, 0)",USCan,143151000.0,124.0,1320600.0
7648,Action,2020,"(Adil El Arbi, 0)","(Peter Craig, 0)","(Will Smith, 2)",USCan,426505244.0,124.0,924000.0
7649,Action,2020,"(Jeff Fowler, 0)","(Pat Casey, 0)","(Ben Schwartz, 0)",USCan,319715683.0,99.0,663000.0
7650,Adventure,2020,"(Stephen Gaghan, 0)","(Stephen Gaghan, 0)","(Robert Downey Jr., 1)",USCan,245487753.0,101.0,296800.0


## Binary Encoding

In [19]:
countrys = ["USCan", "GB", "Oceania", "EU", "East Asia", "Other"]
df["country"] = df["country"].apply(lambda country: [1 if country == c else 0 for c in countrys])

In [20]:
genres = ["Comedy", "Action", "Drama", "Crime", "Biography", "Adventure", "Animation", "Horror", "Fantasy", "Mystery", "Thriller", "Family", "Romance", "Sci-Fi", "Music"]
df["genre"] = df["genre"].apply(lambda genre: [1 if genre == g else 0 for g in genres])

## Normalization

### Continuous

In [21]:
gross = list(df["gross"].sort_values(ascending=False))
df["gross"] = df["gross"].apply(lambda value: round(value/gross[0], 2))

In [22]:
popularityScore = list(df["popularityScore"].sort_values(ascending=False))
df["popularityScore"] = df["popularityScore"].apply(lambda value: round(value/popularityScore[0], 2))

In [23]:
runtime = list(df["runtime"].sort_values(ascending=False))
df["runtime"] = df["runtime"].apply(lambda value: round(value/runtime[0], 2))

In [24]:
year = list(df["year"].sort_values(ascending=False))
diff = year[0] - year[-1]
df["year"] = df["year"].apply(lambda value: round((year[0]-value)/diff, 2))

### Discrete

In [25]:
# remember now that these are in 4 buckets of popularity
# normalize using 4 buckets

df["writer"] = df["writer"].apply(lambda x: round(x[1]/4, 2))
df["star"] = df["star"].apply(lambda x: round(x[1]/4, 2))
df["director"] = df["director"].apply(lambda x: round(x[1]/4, 2))

In [26]:
df

,genre,year,director,writer,star,country,gross,runtime,popularityScore
0,"[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1.0,0.0,0.50,0.25,"[0, 1, 0, 0, 0, 0]",0.02,0.54,0.35
1,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1.0,0.0,0.00,0.00,"[1, 0, 0, 0, 0, 0]",0.02,0.38,0.02
2,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1.0,0.0,0.00,0.00,"[1, 0, 0, 0, 0, 0]",0.19,0.46,0.47
3,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1.0,0.0,0.00,0.00,"[1, 0, 0, 0, 0, 0]",0.03,0.32,0.08
4,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1.0,0.0,0.00,0.25,"[1, 0, 0, 0, 0, 0]",0.01,0.36,0.04
...,...,...,...,...,...,...,...,...,...
7646,"[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.0,0.0,0.25,0.00,"[1, 0, 0, 0, 0, 0]",0.05,0.46,0.06
7648,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.0,0.0,0.00,0.50,"[1, 0, 0, 0, 0, 0]",0.15,0.46,0.04
7649,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.0,0.0,0.00,0.00,"[1, 0, 0, 0, 0, 0]",0.11,0.37,0.03
7650,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.0,0.0,0.00,0.25,"[1, 0, 0, 0, 0, 0]",0.09,0.37,0.01


# Movie Recommender

## Create similarity matrix

In [27]:
def cosine_sim(a, b):
    # Calculate cosine similarity between two vectors
    return round(np.dot(a, b) / (norm(a) * norm(b)), 3)

def flatten_data(row):
    return np.array([item for idx in row for item in (idx if isinstance(idx, list) else [idx])])

#takes 12.5 minutes
# 11.5 minutes

if not os.path.isfile("../data/cosine_data/external/similarity.csv"):
    # Precompute the flattened data
    precomputed_data = [flatten_data(row) for row in df.values]

    # Initialize an empty matrix with the same shape as df
    matrix = []

    for row in range(len(df)):
        if row % 1000 == 0:
            print(row)
        # Calculate cosine similarity for each pair of rows and build row_list using a list comprehension
        row_list = [cosine_sim(precomputed_data[row], precomputed_data[col]) for col in range(len(df))]
        matrix.append(row_list)

    similarity_matrix_df = pd.DataFrame(matrix)
else:
    similarity_matrix_df = pd.read_csv("../data/cosine_data/external/similarity.csv", header=None)

# turn it into a np array
similarity_matrix=np.array(similarity_matrix_df)

## Run inference

In [28]:
movies_liked = np.random.choice(np.array(movie_names), 3)
print(movies_liked)

# movies_liked = []

['Barb Wire(1996)' 'The River Wild(1994)' 'The Santa Clause 2(2002)']


### Similar to individual liked movies

In [29]:
# List of movies enjoyed (chosen from this list: https://docs.google.com/spreadsheets/d/1aLpwOlVCnS4qS3lUn5OKklVW23UXmu1Vb5on0MNuAEw/edit?usp=sharing)

# Major movies from 1980-2020 should be included
print("Movies Liked:","\n")
print(movies_liked, "\n")
print("Recommendations:","\n")
###############

indices = []
for movie in movies_liked:
  indices.append(movie_names.index(movie))

best_recs = []
similarities = {} ## movieid : [similarity, movieid(original)]
for movie_i in indices:

  tmp = similarity_matrix[int(movie_i),:]
  highest_vals = sorted(tmp,reverse=True)[1:4]


  for val in highest_vals:
    new_best_recs = np.where(tmp==val)[0]
    best_recs.extend(new_best_recs)
    for rec in new_best_recs:
      rating = round(100*val,2)
      if rec in similarities and rating>similarities[rec][0]:
        similarities[rec]=[rating,movie_i]
      elif rec not in similarities:
        similarities[rec]=[rating,movie_i]

similarities_sorted = dict(sorted(similarities.items(), key=lambda x:x[1], reverse=True))


for rec in similarities_sorted:
  if movie_names[rec] not in movies_liked:
    print(movie_names[rec]+"............"+str(similarities[rec][0])+"% match to "+movie_names[similarities[rec][1]])


Movies Liked: 

['Barb Wire(1996)' 'The River Wild(1994)' 'The Santa Clause 2(2002)'] 

Recommendations: 

Chocolat(2000)............99.8% match to The Santa Clause 2(2002)
Father of the Bride Part II(1995)............99.8% match to Barb Wire(1996)
Life(1999)............99.8% match to Barb Wire(1996)
The Replacements(2000)............99.8% match to Barb Wire(1996)
Keeping the Faith(2000)............99.8% match to Barb Wire(1996)
Dr. T & the Women(2000)............99.8% match to Barb Wire(1996)
28 Days Later...(2002)............99.7% match to The Santa Clause 2(2002)
Possession(2002)............98.7% match to The Santa Clause 2(2002)
Code 46(2003)............98.7% match to The Santa Clause 2(2002)
Anonymous(2011)............98.7% match to The Santa Clause 2(2002)
Home Alone 2: Lost in New York(1992)............98.5% match to The River Wild(1994)
National Lampoon's European Vacation(1985)............98.3% match to The River Wild(1994)
Stand by Me(1986)............98.1% match to The River

### Similar to all liked movies

In [30]:
def movie_recommendations(movies_liked, similarity_matrix, movie_names):
    indices = [movie_names.index(movie) for movie in movies_liked]

    best_recs = []
    similarities = {}
    similarity_row = np.mean([similarity_matrix[i] for i in indices], axis=0)

    highest_vals = sorted(similarity_row, reverse=True)[1:10]

    for val in highest_vals:
        new_best_recs = np.where(similarity_row==val)[0]
        best_recs.extend(new_best_recs)
        for rec in new_best_recs:
            similarities[rec] = max(similarities.get(rec, 0), 100*val)

    similarities_sorted = dict(sorted(similarities.items(), key=lambda x:x[1], reverse=True))

    rec_list = []
    for rec in similarities_sorted:
        if movie_names[rec] not in movies_liked:
            rec_list.append(f"{movie_names[rec]}...{round(similarities[rec], 2)}% match")

    return rec_list

# Print recommendations
print("Movies most similar to all the liked movies:","\n")
print(*movie_recommendations(movies_liked, similarity_matrix, movie_names), sep='\n')

Movies most similar to all the liked movies: 

Ironweed(1987)...60.53% match
Disclosure(1994)...60.37% match
Havana(1990)...60.23% match
A Chorus Line(1985)...60.2% match
JFK(1991)...60.2% match
Platoon(1986)...60.17% match
9½ Weeks(1986)...60.17% match
An Officer and a Gentleman(1982)...60.13% match
Jungle Fever(1991)...60.13% match


## Save Dataframes to csv

In [31]:
if not os.path.isfile("../data/cosine_data/external/movie_names.csv"):
    movie_names_df = pd.DataFrame(movie_names)


    processed_dir = "../data/cosine_data/external"
    if os.path.exists(processed_dir):
        processed_path = processed_dir + "/movie_names.csv"
        movie_names_df.to_csv(processed_path, index=False)
        print("Saved movie names dataset: %s", processed_path)
    else:
        print("path does not exist")
else:
    print("file already exists")

file already exists


In [32]:
if not os.path.isfile("../data/cosine_data/external/similarity.csv"):

    processed_dir = "../data/cosine_data/external"
    if os.path.exists(processed_dir):
        processed_path = processed_dir + "/similarity.csv"
        similarity_matrix_df.to_csv(processed_path, index=False)
        print("Saved similarity dataset: %s", processed_path)
    else:
        print("path does not exist")
else:
    print("file already exists")

file already exists


In [33]:
if not os.path.isfile("../data/cosine_data/processed/movies_processed.csv"):

    processed_dir = "../data/cosine_data/processed"
    if os.path.exists(processed_dir):
        processed_path = processed_dir + "/movies_processed.csv"
        df.to_csv(processed_path, index=False)
        print("Saved processed movies dataset: %s", processed_path)
    else:
        print("path does not exist")
else:
    print("file already exists")

file already exists
